# Comparativo global: MASTER, TFB, RandomTopJ e benchmarks

Consolida os resultados globais e compara benchmarks, modelos TFB e modelos MASTER.

A comparação principal deve ser feita por `janela_trading`/`horizonte_comparavel` (`k` de rebalanceamento), não apenas por `pred_len`.


In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'utils':
    ROOT = ROOT.parent
if ROOT.name != 'paralelo' and (ROOT / 'paralelo').exists():
    ROOT = ROOT / 'paralelo'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.comparativo_metricas_global import (
    comparar_global,
    METRICAS_INTERESSE,
    resumo_por_modelo,
    top_configs as gerar_top_configs,
    ranking_global,
)

OUT = ROOT / 'simulacoes' / 'comparativo_global_master_tfb'
OUT


In [ ]:
dfs = comparar_global(
    base_dir=ROOT,
    output_dir='simulacoes/comparativo_global_master_tfb',
    top_n=30,
)

metricas = dfs['metricas'].copy()
print(f'Linhas carregadas antes dos ajustes: {len(metricas)}')
print(f'Arquivos salvos em: {OUT}')
metricas.head()


In [ ]:
def simplificar_dataset_master(nome):
    nome = str(nome)
    if re.search(r'(__|_|-)log_returns?$', nome):
        return 'log_return'
    if re.search(r'(__|_|-)returns?$', nome):
        return 'return'
    if re.search(r'(__|_|-)prices?$', nome):
        return 'prices'
    return nome

def corrigir_datasets_master(df):
    if df is None or df.empty or 'dataset' not in df.columns:
        return df
    df = df.copy()
    if 'grupo' in df.columns:
        mask = df['grupo'].eq('MASTER')
        df.loc[mask, 'dataset'] = df.loc[mask, 'dataset'].apply(simplificar_dataset_master)
    else:
        df['dataset'] = df['dataset'].apply(simplificar_dataset_master)
    return df

def output_dir_relativo(json_path):
    parent = Path(json_path).parent
    if parent.is_absolute():
        try:
            return str(parent.relative_to(ROOT))
        except ValueError:
            return str(parent)
    return str(parent)

def anexar_acerto_negativo_master(df):
    df = df.copy()
    if df.empty or 'json_path' not in df.columns:
        return df

    neg_path = ROOT / 'simulacoes' / 'master_tfb_experimento' / 'acerto_negativos_master.csv'
    if not neg_path.exists():
        print(f'Aviso: arquivo não encontrado: {neg_path}')
        return df

    neg_master = pd.read_csv(neg_path)
    df['output_dir'] = df['json_path'].apply(output_dir_relativo)

    cols_neg = [
        'output_dir',
        'taxa_acerto_negativos',
        'mean_precision_negative',
        'n_pred_negativos',
        'n_acertos_negativos',
        'n_janelas_com_negativos',
    ]
    cols_neg = [c for c in cols_neg if c in neg_master.columns]

    df = df.drop(columns=[c for c in cols_neg if c != 'output_dir'], errors='ignore')
    df = df.merge(
        neg_master[cols_neg].drop_duplicates('output_dir'),
        on='output_dir',
        how='left',
    )
    return df

metricas = anexar_acerto_negativo_master(metricas)
metricas = corrigir_datasets_master(metricas)

# Recalcula tabelas derivadas após incluir os negativos do MASTER.
resumo = corrigir_datasets_master(resumo_por_modelo(metricas))
top_configs = corrigir_datasets_master(gerar_top_configs(metricas, n=30))
ranking = corrigir_datasets_master(ranking_global(metricas))

print(f'Linhas carregadas após os ajustes: {len(metricas)}')
metricas[[c for c in [
    'grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading',
    'mean_spearman_ic', 'mean_precision_positive', 'mean_precision_negative', 'output_dir'
] if c in metricas.columns]].head()


In [ ]:
# Cobertura por grupo/modelo
if not metricas.empty:
    display(metricas.groupby('grupo').size().rename('n_resultados').reset_index())
    display(
        metricas.groupby(['grupo', 'modelo'])
        .size()
        .rename('n_resultados')
        .reset_index()
        .sort_values(['grupo', 'n_resultados'], ascending=[True, False])
        .head(50)
    )


In [ ]:
# Tabela principal das métricas de interesse
cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', *METRICAS_INTERESSE, 'json_path']
tabela = metricas[[c for c in cols if c in metricas.columns]].copy()
tabela.sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last').head(50)


In [ ]:
# Melhores configurações por métrica
top_configs.head(80)


In [ ]:
# Ranking médio simples nas métricas de interesse: quanto menor, melhor.
ranking_cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', 'rank_medio_metricas_interesse', *METRICAS_INTERESSE, 'json_path']
ranking[[c for c in ranking_cols if c in ranking.columns]].head(50)


In [ ]:
# Resumo agregado por grupo/modelo/janela de trading
resumo.sort_values(['janela_trading', 'mean_spearman_ic_mean'], ascending=[True, False], na_position='last').head(80)


In [ ]:
# Comparação direta por janela de trading e modelo, usando medianas
if not metricas.empty:
    comp = (
        metricas
        .groupby(['janela_trading', 'grupo', 'modelo'], dropna=False)[[c for c in METRICAS_INTERESSE if c in metricas.columns]]
        .median()
        .reset_index()
        .sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last')
    )
    display(comp.head(100))


## Análise gráfica comparativa

Gráficos para comparar como o `mean_spearman_ic` e as precisões positiva/negativa variam com a janela de trading.


In [ ]:
def base_agregada_metrica(df, metrica, agg='median'):
    if df.empty or metrica not in df.columns:
        return pd.DataFrame()
    base = df.dropna(subset=['janela_trading', metrica]).copy()
    base['janela_trading'] = pd.to_numeric(base['janela_trading'], errors='coerce')
    base = base.dropna(subset=['janela_trading'])
    if base.empty:
        return base
    return (
        base.groupby(['janela_trading', 'grupo', 'modelo'], dropna=False)[metrica]
        .agg(agg)
        .reset_index()
        .assign(serie=lambda x: x['grupo'].astype(str) + ' | ' + x['modelo'].astype(str))
    )

def plot_metrica_por_janela(df, metrica, titulo, ylabel=None, top_n=14, hline_zero=False):
    base = base_agregada_metrica(df, metrica)
    if base.empty:
        print(f'Sem dados para {metrica}.')
        return None
    ordem = base.groupby('serie')[metrica].median().sort_values(ascending=False).head(top_n).index
    base = base[base['serie'].isin(ordem)]

    fig, ax = plt.subplots(figsize=(12, 5))
    for serie, g in base.groupby('serie'):
        g = g.sort_values('janela_trading')
        ax.plot(g['janela_trading'], g[metrica], marker='o', label=serie)
    if hline_zero:
        ax.axhline(0, linestyle='--', linewidth=1)
    ax.set_xlabel('janela de trading / k')
    ax.set_ylabel(ylabel or metrica)
    ax.set_title(titulo)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()
    return base

def plot_ranking_metrica(df, metrica, titulo, top_n=20):
    if df.empty or metrica not in df.columns:
        print(f'Sem dados para {metrica}.')
        return None
    base = (
        df.dropna(subset=[metrica])
        .groupby(['grupo', 'modelo'], dropna=False)[metrica]
        .median()
        .reset_index()
    )
    if base.empty:
        print(f'Sem dados para {metrica}.')
        return None
    base['serie'] = base['grupo'].astype(str) + ' | ' + base['modelo'].astype(str)
    base = base.sort_values(metrica, ascending=False).head(top_n).sort_values(metrica)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(base))))
    ax.barh(base['serie'], base[metrica])
    ax.set_xlabel(metrica)
    ax.set_title(titulo)
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    return base


In [ ]:
base_ic_janela = plot_metrica_por_janela(
    metricas, 'mean_spearman_ic', 'Spearman IC mediano por janela de trading',
    ylabel='mean_spearman_ic mediano', hline_zero=True
)
base_prec_pos_janela = plot_metrica_por_janela(
    metricas, 'mean_precision_positive', 'Precisão positiva mediana por janela de trading',
    ylabel='mean_precision_positive mediana'
)
base_prec_neg_janela = plot_metrica_por_janela(
    metricas, 'mean_precision_negative', 'Precisão negativa mediana por janela de trading',
    ylabel='mean_precision_negative mediana'
)

rank_ic = plot_ranking_metrica(metricas, 'mean_spearman_ic', 'Top modelos por Spearman IC mediano')
rank_pos = plot_ranking_metrica(metricas, 'mean_precision_positive', 'Top modelos por precisão positiva mediana')
rank_neg = plot_ranking_metrica(metricas, 'mean_precision_negative', 'Top modelos por precisão negativa mediana')


## Arquivos gerados

- `metricas_global_long.csv`: base longa, uma linha por JSON encontrado.
- `tabela_metricas_interesse.csv`: somente as métricas centrais.
- `resumo_por_modelo.csv`: média/mediana/desvio/contagem por grupo, dataset, modelo e janela de trading.
- `top_configs_por_metrica.csv`: melhores configurações por métrica.
- `ranking_global.csv`: ranking médio simples das métricas de interesse.
